In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms

import onnx
import tensorflow as tf

In [2]:
class SimpleCNN(nn.Module):

    def __init__(self):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=8,
            kernel_size=3
        )

        self.relu = nn.ReLU()

        self.pool = nn.MaxPool2d(2)

        self.fc = nn.Linear(8 * 13 * 13, 10)

    def forward(self, x):

        x = self.conv1(x)

        x = self.relu(x)

        x = self.pool(x)

        x = x.view(x.size(0), -1)

        x = self.fc(x)

        return x

In [3]:
model = SimpleCNN()

print(model)

SimpleCNN(
  (conv1): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1))
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=1352, out_features=10, bias=True)
)


In [4]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

100%|█████████████████████████████████████| 9.91M/9.91M [00:06<00:00, 1.62MB/s]
100%|█████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 93.8kB/s]
100%|██████████████████████████████████████| 1.65M/1.65M [00:06<00:00, 268kB/s]
100%|██████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 479kB/s]


In [5]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [6]:
model.train()

for epoch in range(1):

    running_loss = 0.0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.4023


In [7]:
torch.save(model.state_dict(), "simple_cnn.pth")

In [8]:
import os

print(os.path.exists("simple_cnn.pth"))

True


In [9]:
model.eval()

SimpleCNN(
  (conv1): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1))
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=1352, out_features=10, bias=True)
)

In [10]:
dummy_input = torch.randn(1, 1, 28, 28)

In [11]:
import onnx

onnx_model = onnx.load("simple_cnn.onnx")
onnx.checker.check_model(onnx_model)

print("ONNX model is valid!")

ONNX model is valid!


In [12]:
torch.onnx.export(
    model,
    dummy_input,
    "simple_cnn.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=18
)

[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


/opt/anaconda3/envs/quantization/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.13.0',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[1,1,28,28]>
            ),
            outputs=(
                %"output"<FLOAT,[1,10]>
            ),
            initializers=(
                %"conv1.weight"<FLOAT,[8,1,3,3]>{TorchTensor(...)},
                %"conv1.bias"<FLOAT,[8]>{TorchTensor<FLOAT,[8]>(Parameter containing: tensor([ 0.0232,  0.0017,  0.3257, -0.0004,  0.0785, -0.2106, -0.1158,  0.0128], requires_grad=True), name='conv1.bias')},
                %"fc.weight"<FLOAT,[10,1352]>{TorchTensor(...)},
                %"fc.bias"<FLOAT,[10]>{TorchTensor<FLOAT,[10]>(Parameter containing: tensor([-0.0348,  0.0418,  0.0100, -0.0286, -0.0095,  0.0335, -0.0314,  0.0323, -0.0166,  0.0293], requir

In [16]:
import tensorflow as tf

model = tf.saved_model.load("saved_model")

print("TensorFlow SavedModel loaded successfully!")

TensorFlow SavedModel loaded successfully!


In [17]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_saved_model("saved_model")

In [18]:
converter.optimizations = [tf.lite.Optimize.DEFAULT]

In [19]:
import numpy as np

def representative_dataset():
    count = 0

    for images, labels in train_loader:

        images = images.numpy()            # (64,1,28,28)

        for img in images:

            img = np.transpose(img, (1, 2, 0))   # (28,28,1)

            img = np.expand_dims(img, axis=0)    # (1,28,28,1)

            yield [img.astype(np.float32)]

            count += 1

            if count >= 100:
                return

In [20]:
converter.representative_dataset = representative_dataset

In [21]:
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

In [22]:
tflite_model = converter.convert()

W0000 00:00:1784907892.479225   64698 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1784907892.479241   64698 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1784907892.479707   64698 reader.cc:83] Reading SavedModel from: saved_model
I0000 00:00:1784907892.479876   64698 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1784907892.479880   64698 reader.cc:147] Reading SavedModel debug info (if present) from: saved_model
I0000 00:00:1784907892.480616   64698 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1784907892.480727   64698 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1784907892.484109   64698 loader.cc:220] Running initialization op on SavedModel bundle at path: saved_model
I0000 00:00:1784907892.485651   64698 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 5945 microseconds.
I0000 00:00:1784907892.489217   64698 dump_mlir_util.

In [23]:
with open("simple_cnn_int8.tflite", "wb") as f:
    f.write(tflite_model)

print("INT8 TFLite model saved successfully!")

INT8 TFLite model saved successfully!


In [24]:
import os

print(os.path.exists("simple_cnn_int8.tflite"))


True


In [25]:
import os
import numpy as np

os.makedirs("calib", exist_ok=True)

count = 0

for images, labels in train_loader:

    images = images.numpy()

    for img in images:

        img = np.transpose(img, (1,2,0))

        np.save(f"calib/sample_{count}.npy", img)

        count += 1

        if count == 100:
            break

    if count == 100:
        break

print("Saved", count, "calibration images")

Saved 100 calibration images
